# Configurable Regressor Experiments

This notebook is organized so users can run experiments on their own data by editing only `CONFIG`.

Workflow:
1. Update paths and feature options in `CONFIG`.
2. Run all cells.
3. Check saved metrics/plots under `plots/<experiment_name>@pca<k>/`.


In [21]:
%matplotlib inline

import json
import pandas as pd


## Main Config

Configure json file to run experiments on a different dataset.


In [22]:
config_path = 'configs/default.json'
with open(config_path, "r") as f:
    CONFIG = json.load(f)
CONFIG

{'seed': 42,
 'experiment_name': 'baseline',
 'test_size': 0.2,
 'scale_features': True,
 'permutation_repeats': 10,
 'save_dataset_snapshot': True,
 'save_mean_correlations': True,
 'save_best_correlations': False,
 'correlation_metrics': {'enabled': True,
  'items': [{'label': 'MUSIQ (NR)',
    'feature': 'nr',
    'metric': 'musiq',
    'higher_is_better': True},
   {'label': 'LPIPS-VGG+GT (FR)',
    'feature': 'fr',
    'metric': 'lpips-vgg',
    'reference': 'gt',
    'higher_is_better': False},
   {'label': 'ARNIQA (NR)',
    'feature': 'nr',
    'metric': 'arniqa',
    'higher_is_better': True}]},
 'paths': {'raw_scores': 'scores/labels.csv',
  'scores': 'scores/fn_scores.csv',
  'features_root': 'features',
  'plots_root': 'plots'},
 'score_preparation': {'enabled': True,
  'method_column': 'method',
  'case_column': 'test_case',
  'score_column': 'score_norm',
  'method_map': {'pasd': 'PASD', 'supir': 'SUPIR', 'realesrgan': 'RealESRGAN'},
  'name_suffix': '.npy.gz'},
 'dataset

## Helpers

The reusable experiment code lives in `qualisr_lab.regressors`, so notebook and CLI runs stay in sync.


In [23]:
from qualisr_lab.regressors import deep_update, run_experiment


## Run One Experiment

In [24]:
run = run_experiment(CONFIG)
run["results"]


,model,plcc,srcc,source,feature,column,higher_is_better,normalized
0,catboost,0.945747,0.946957,regressor,NaN,NaN,NaN,NaN
1,randomforest,0.883573,0.918261,regressor,NaN,NaN,NaN,NaN
2,mean,0.880559,0.883768,summary,NaN,NaN,NaN,NaN
3,xgb,0.812356,0.786087,regressor,NaN,NaN,NaN,NaN
4,MUSIQ (NR),0.738571,0.747826,metric,nr,musiq,True,True
5,ARNIQA (NR),0.541670,0.466957,metric,nr,arniqa,True,True
6,LPIPS-VGG+GT (FR),0.305958,0.247826,metric,fr,lpips-vgg_gt,False,True


## Optional Batch Runs

Set overrides and run several experiments in one go.


In [25]:
# Override config loaded before
BATCH_OVERRIDES = [
    # {
    #     "experiment_name": "nr_only",
    #     "features": {
    #         "include": ["nr"],
    #         "include_stats": False,
    #     },
    # },
    # {
    #     "experiment_name": "fr_vgg_resnet",
    #     "features": {
    #         "include": ["fr", "vgg", "resnet"],
    #         "fr_refs": ["rlfn", "span"],
    #     },
    # },
]

# Launch alternate configs
CONFIG_FILES = [
    # "configs/default.json"
]

batch_results = []
for override in BATCH_OVERRIDES:
    cfg = deep_update(CONFIG, override)
    result = run_experiment(cfg)
    row = result["results"].copy()
    row.insert(0, "experiment", cfg["experiment_name"])
    batch_results.append(row)
for config_path in CONFIG_FILES:
    with open(config_path, "r") as f:
        cfg = json.load(f)
    result = run_experiment(cfg)
    row = result["results"].copy()
    row.insert(0, "experiment", cfg["experiment_name"])
    batch_results.append(row)

if batch_results:
    summary = pd.concat(batch_results, ignore_index=True)
    display(summary)
else:
    print("No batch overrides configured.")


No batch overrides configured.
